# Data Understanding

## Objective
Load the Bank Marketing dataset and perform an initial structural inspection:
- Confirm row count, column count, and column names
- Inspect data types
- Identify the target variable and its distribution
- Check for missing values and duplicates
- Flag potential leakage variables

In [1]:
import pandas as pd
import numpy as np

# Reproducibility
RANDOM_STATE = 42

df = pd.read_csv('../data/raw/bank_marketing.csv')
print(f"Shape: {df.shape}")
df.head()

Shape: (45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN,no
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN,no
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN,no
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN,no


## Column names and data types

In [2]:
df.dtypes

age            int64
job              str
marital          str
education        str
default          str
balance        int64
housing          str
loan             str
contact          str
day_of_week    int64
month            str
duration       int64
campaign       int64
pdays          int64
previous       int64
poutcome         str
y                str
dtype: object

## Missing values

In [3]:
missing = df.isnull().sum()
print("NaN missing values:")
print(missing[missing > 0])

print("\n'unknown' string counts per categorical column:")
for col in df.select_dtypes(include='object').columns:
    n = (df[col] == 'unknown').sum()
    if n > 0:
        print(f"  {col}: {n} ({n/len(df)*100:.1f}%)")

NaN missing values:
job            288
education     1857
contact      13020
poutcome     36959
dtype: int64

'unknown' string counts per categorical column:


C:\Users\UsEr\AppData\Local\Temp\ipykernel_4300\1842834209.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:


## Duplicate rows

In [4]:
dupes = df.duplicated().sum()
print(f"Exact duplicate rows: {dupes}")

Exact duplicate rows: 0


## Target variable

In [5]:
print("Target distribution:")
print(df['y'].value_counts())
print()
print("Target distribution (%):")
print(df['y'].value_counts(normalize=True).mul(100).round(1))

Target distribution:
y
no     39922
yes     5289
Name: count, dtype: int64

Target distribution (%):
y
no     88.3
yes    11.7
Name: proportion, dtype: float64


## Unique values per column

In [6]:
for col in df.columns:
    print(f"{col}: {df[col].nunique()} unique values")

age: 77 unique values
job: 11 unique values
marital: 3 unique values
education: 3 unique values
default: 2 unique values
balance: 7168 unique values
housing: 2 unique values
loan: 2 unique values
contact: 2 unique values
day_of_week: 31 unique values
month: 12 unique values
duration: 1573 unique values
campaign: 48 unique values
pdays: 559 unique values
previous: 41 unique values
poutcome: 3 unique values
y: 2 unique values


## Key Findings

| Metric | Value |
|--------|-------|
| Rows | 45,211 |
| Columns | 17 |
| Target | `y` (binary: yes / no) |
| Positive class (`yes`) | 5,289 (~11.7%) |
| Negative class (`no`) | 39,922 (~88.3%) |
| Exact duplicates | 0 |
| NaN missing values | Present in `job`, `education`, `contact` |
| 'unknown' categories | Present in `job`, `education`, `contact`, `poutcome` |
| Potential leakage | `duration` — only known after the call ends |

## Decisions Made

- The target `y` is binary → binary classification task.
- `duration` will be excluded due to target leakage (it is only known after the outcome).
- `'unknown'` string values are not NaN; they will be treated carefully in preprocessing.
- No duplicate rows found — no deduplication required.